# TAREFA 5
Desenvolvimento Rápido de Aplicações em Python

> Marcos Alcino Ribeiro Cussioli

> 202402394612

# Gerenciador de Alunos e Notas com Interface Gráfica para Google Colab

Este notebook contém uma aplicação em Python para gerenciar alunos e suas notas, com uma interface gráfica interativa construída usando `ipywidgets`, ideal para ser executada no ambiente do Google Colab.

## Funcionalidades:
*   **Cadastrar Aluno:** Adiciona novos alunos com matrícula e nome.
*   **Adicionar Nota:** Registra notas para disciplinas de alunos existentes.
*   **Calcular Média:** Calcula e exibe a média das notas de um aluno.
*   **Salvar/Carregar Dados:** Persiste os dados dos alunos em um arquivo JSON (`dados_alunos.json`) e os carrega de volta.
*   **Exibir Alunos:** Lista todos os alunos cadastrados com suas respectivas notas.

## Como usar:

1.  **Execute as células:** Execute todas as células deste notebook em sequência.
2.  **Interaja com a GUI:** Após a execução da última célula, a interface gráfica será exibida abaixo dela. Use os campos de entrada e botões para interagir com a aplicação.
3.  **Persistência de Dados:** Os dados serão salvos e carregados automaticamente do arquivo `dados_alunos.json` no ambiente do Colab. Você pode baixar este arquivo a qualquer momento para ter um backup dos seus dados.


In [ ]:
# Instalação da biblioteca ipywidgets (se necessário)
!pip install ipywidgets

In [ ]:
import json
from IPython.display import display, HTML
import ipywidgets as widgets
from ipywidgets import Layout, Button, Box, Text, FloatText, VBox, HBox, Output

# Dicionário global para armazenar os dados dos alunos
alunos = {}
# Nome do arquivo para persistência dos dados
ARQUIVO_DADOS = "dados_alunos.json"

# Output widget para exibir mensagens
output_area = Output(layout={'border': '1px solid black'})

def cadastrar_aluno_gui(matricula_input, nome_input):
    with output_area:
        output_area.clear_output()
        matricula = matricula_input.value.strip()
        nome = nome_input.value.strip()

        if not matricula or not nome:
            print("Erro: Matrícula e nome não podem ser vazios.")
            return

        if matricula in alunos:
            print("Erro: Matrícula já existe.")
            return

        alunos[matricula] = {"nome": nome, "disciplinas": []}
        print(f"Aluno {nome} ({matricula}) cadastrado com sucesso.")
        matricula_input.value = ""
        nome_input.value = ""

def adicionar_nota_gui(matricula_input, disciplina_input, nota_input):
    with output_area:
        output_area.clear_output()
        matricula = matricula_input.value.strip()
        disciplina = disciplina_input.value.strip()

        if not matricula or not disciplina:
            print("Erro: Matrícula e disciplina não podem ser vazios.")
            return

        if matricula not in alunos:
            print("Erro: Aluno não encontrado.")
            return

        try:
            nota = float(nota_input.value)
            if not (0 <= nota <= 10):
                raise ValueError("A nota deve estar entre 0 e 10.")
        except ValueError as e:
            print(f"Erro: {e}. Por favor, digite um número válido.")
            return

        alunos[matricula]["disciplinas"].append((disciplina, nota))
        print(f"Nota {nota} adicionada à disciplina {disciplina} para o aluno {alunos[matricula]['nome']}.")
        matricula_input.value = ""
        disciplina_input.value = ""
        nota_input.value = 0.0

def calcular_media_gui(matricula_input):
    with output_area:
        output_area.clear_output()
        matricula = matricula_input.value.strip()

        if not matricula:
            print("Erro: Matrícula não pode ser vazia.")
            return

        if matricula not in alunos:
            print("Erro: Aluno não encontrado.")
            return

        disciplinas = alunos[matricula]["disciplinas"]
        if not disciplinas:
            print(f"O aluno {alunos[matricula]['nome']} não possui notas registradas.")
            return

        total_notas = sum(nota for _, nota in disciplinas)
        media = total_notas / len(disciplinas)
        print(f"A média do aluno {alunos[matricula]['nome']} é: {media:.2f}")
        matricula_input.value = ""

def salvar_dados_gui():
    with output_area:
        output_area.clear_output()
        try:
            with open(ARQUIVO_DADOS, "w", encoding="utf-8") as f:
                dados_para_salvar = {mat: {"nome": aluno["nome"], "disciplinas": [list(d) for d in aluno["disciplinas"]]} for mat, aluno in alunos.items()}
                json.dump(dados_para_salvar, f, indent=4)
            print("Dados salvos com sucesso.")
        except IOError as e:
            print(f"Erro ao salvar dados: {e}")

def carregar_dados_gui():
    global alunos
    with output_area:
        output_area.clear_output()
        try:
            with open(ARQUIVO_DADOS, "r", encoding="utf-8") as f:
                dados_carregados = json.load(f)
                alunos = {mat: {"nome": aluno["nome"], "disciplinas": [tuple(d) for d in aluno["disciplinas"]]} for mat, aluno in dados_carregados.items()}
            print("Dados carregados com sucesso.")
        except FileNotFoundError:
            print("Arquivo de dados não encontrado. Iniciando com dados vazios.")
            alunos = {}
        except json.JSONDecodeError:
            print("Erro ao decodificar o arquivo JSON. Iniciando com dados vazios.")
            alunos = {}
        except Exception as e:
            print(f"Ocorreu um erro inesperado ao carregar os dados: {e}")
            alunos = {}

def exibir_alunos_gui():
    with output_area:
        output_area.clear_output()
        if not alunos:
            print("Nenhum aluno cadastrado.")
            return
        print("--- Alunos Cadastrados ---")
        for matricula, dados in alunos.items():
            print(f"Matrícula: {matricula}, Nome: {dados['nome']}")
            if dados['disciplinas']:
                for disciplina, nota in dados['disciplinas']:
                    print(f"  - {disciplina}: {nota}")
            else:
                print("  (Sem notas)")

def build_gui():
    # Carregar dados ao iniciar a GUI
    carregar_dados_gui()

    # --- Cadastrar Aluno ---
    matricula_cadastro = Text(description='Matrícula:', layout=Layout(width='auto'))
    nome_cadastro = Text(description='Nome:', layout=Layout(width='auto'))
    btn_cadastrar = Button(description='Cadastrar Aluno', button_style='success', layout=Layout(width='auto'))
    btn_cadastrar.on_click(lambda b: cadastrar_aluno_gui(matricula_cadastro, nome_cadastro))
    cadastro_box = VBox([matricula_cadastro, nome_cadastro, btn_cadastrar])

    # --- Adicionar Nota ---
    matricula_nota = Text(description='Matrícula:', layout=Layout(width='auto'))
    disciplina_nota = Text(description='Disciplina:', layout=Layout(width='auto'))
    nota_nota = FloatText(description='Nota (0-10):', min=0.0, max=10.0, step=0.1, layout=Layout(width='auto'))
    btn_adicionar_nota = Button(description='Adicionar Nota', button_style='info', layout=Layout(width='auto'))
    btn_adicionar_nota.on_click(lambda b: adicionar_nota_gui(matricula_nota, disciplina_nota, nota_nota))
    nota_box = VBox([matricula_nota, disciplina_nota, nota_nota, btn_adicionar_nota])

    # --- Calcular Média ---
    matricula_media = Text(description='Matrícula:', layout=Layout(width='auto'))
    btn_calcular_media = Button(description='Calcular Média', button_style='warning', layout=Layout(width='auto'))
    btn_calcular_media.on_click(lambda b: calcular_media_gui(matricula_media))
    media_box = VBox([matricula_media, btn_calcular_media])

    # --- Salvar/Carregar/Exibir ---
    btn_salvar = Button(description='Salvar Dados', button_style='primary', layout=Layout(width='auto'))
    btn_salvar.on_click(lambda b: salvar_dados_gui())
    btn_carregar = Button(description='Carregar Dados', button_style='primary', layout=Layout(width='auto'))
    btn_carregar.on_click(lambda b: carregar_dados_gui())
    btn_exibir_alunos = Button(description='Exibir Alunos', button_style='primary', layout=Layout(width='auto'))
    btn_exibir_alunos.on_click(lambda b: exibir_alunos_gui())

    geral_box = HBox([btn_salvar, btn_carregar, btn_exibir_alunos], layout=Layout(justify_content='space-around'))

    # --- Tabs ---
    tab_contents = [
        cadastro_box,
        nota_box,
        media_box,
        geral_box
    ]
    tab = widgets.Tab(children=tab_contents)
    tab.set_title(0, 'Cadastrar Aluno')
    tab.set_title(1, 'Adicionar Nota')
    tab.set_title(2, 'Calcular Média')
    tab.set_title(3, 'Dados/Exibir')

    display(VBox([tab, output_area]))

# Para usar a interface gráfica, chame build_gui() diretamente
build_gui()